### Applying Isolation Forest algorithm to flag unstable satellites
- We will now train Isolation forest on each satellite cluster 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest

import shap # For model O/P reasoning

In [ ]:
# 1. Read the unscaled clustered dataset
unscaled_df = pd.read_csv('../data/05_labeled/satellites_unscaled_labeled.csv')
# 2. Read the scaled clustered dataset
scaled_df = pd.read_csv('../data/05_labeled/satellites_scaled_labeled.csv')

In [ ]:
unscaled_df.head()

---
## 🛰️ Cluster-Wise Isolation Forest: Abstract Workflow

Below is a high-level, implementation-agnostic explanation of how to run
**Isolation Forest separately on each cluster**, with parameters that adapt
to each cluster’s size and spread.

---

### 1. **Prepare Cluster Subsets**
After clustering (e.g., via K-Means), split the dataset into groups:

- Each group represents one cluster.
- Each cluster is treated as its own “local data pocket.”
- Compute basic stats per cluster:
  - `nᵢ` → number of points  
  - `σᵢ` → average standard deviation (or IQR) -> take average sd of every feature

These two values guide all later parameter choices.

---

### 2. **Set Cluster-Specific Parameters**
For every cluster **Cᵢ**:

**a. n_estimators (tree count)**  
Adjust based on cluster size:
- Small clusters → fewer trees  
- Medium → moderate  
- Large → slightly more

**b. max_samples**  
Use the smaller of:
- the cluster size,  
- a fixed cap (traditional ≈ 256)

This keeps trees consistent but efficient.

**c. contamination**  
Link sensitivity to cluster spread:
- Tight/compact clusters → very low contamination  
- Medium spread → moderate  
- High-spread/noisy clusters → slightly higher

This makes each model respect its own local variance.

**d. max_features**  
Use all features inside each cluster.  
Local models benefit from full context.

**e. bootstrap**  
Only enable if the cluster is very large.

---

### 3. **Train a Local Isolation Forest Inside Each Cluster**
- Fit the detector using that cluster’s own data only.
- The model isolates anomalies relative to the internal geometry of the cluster.
- No global interference from other clusters.

This produces cluster-wise anomaly scores.

---

### 4. **Normalize Scores Inside Each Cluster**
Each cluster’s score distribution is different.

Normalize scores cluster-wise, for example:
- min–max scaling  
- or z-score

This ensures that anomaly scores across clusters become comparable.

---

### 5. **Merge All Results**
Bring everything back together:

- Reattach the cluster label  
- Append the normalized anomaly score  
- Combine all cluster outputs into one final dataframe

The final result gives:
- local anomaly detection quality  
- global comparability  
- very stable and interpretable behavior

---

### 🧭 Outcome
You get a lightweight, cluster-sensible anomaly system where:
- each cluster has a detector tuned to its density and size  
- anomaly detection is cleaner and more precise  
- merged scores reflect global anomaly significance without distortion

This structure is simple, scalable, and strong for datasets with natural subgroups
(e.g., orbital shells, behavioral clusters, sensor regimes, etc.).


---


- Check the raw size of the clusters

In [ ]:
scaled_df['CLUSTER'].value_counts()

---
### 1. Basic stats for each cluster -> Use scaled dataset for unbiased Standard Deviation
1. Satellite count
2. Average standard deviation across numerical features

In [ ]:
# Find number of samples and mean SD 
def compute_base_stats(scaled_df):

    cluster_size_list = []
    sd_list = []
    value_counts = scaled_df['CLUSTER'].value_counts()
    
    # For each cluster: compute clust size and average SD
    for clust_id in sorted(value_counts.index):
        # --- clust size ---
        cluster_size_list.append(value_counts.loc[clust_id]) 
        
        # --- Average SD ---
        clust_rows = scaled_df[scaled_df['CLUSTER'] == clust_id] # filter out only this cluster satellites

        numeric_cols = clust_rows.select_dtypes(include = np.number).drop(columns=['CLUSTER']) # get numeric column names
        encoded_cols = [col for col in numeric_cols.columns if col.startswith('SAT_TYPE_')] # Filter out encoded features, as we dont need them for SD
        numeric_cols = numeric_cols.drop(columns = encoded_cols).columns
        
        total_sd = 0 # Find the SD for each feature, then add the SD for the current cluster's all features
        for feature in numeric_cols:
            sd = clust_rows[feature].std()
            total_sd += sd

        avg_sd = total_sd / len(numeric_cols) 
        sd_list.append(avg_sd)
        
    return cluster_size_list, sd_list
        

In [ ]:
# Call the function
cluster_size, avg_sd = compute_base_stats(scaled_df)

In [ ]:
cluster_size

In [ ]:
avg_sd

----
### 2. Setting cluster specific parameters for Isolation Forest models
- For each cluster based on the size and Average SD, find:
1. **n_estimators**: Tree count
2. **max_samples**:  Num of samples to use in each isolation tree
3. **contamination**: Threshold by which anomaly will be selected
4. **max_features**: Use all features in every tree

---
## Mapping Cluster Stats → Isolation Forest Parameters (Intuition + Value Ranges)

### Inputs:
- **cluster_size** → number of satellites in this cluster  
- **cluster_sd** → how spread-out the cluster is (average SD of continuous features)

These two fully define how the Isolation Forest should be shaped.

---

## 1. contamination (comes from cluster_sd)

### Intuition:
“Tight cluster → fewer anomalies. Loose cluster → more anomalies.”

### Suggested values:
- **Low SD** (very tight) → `0.01` to `0.02`  
- **Medium SD** → `0.03` to `0.06`  
- **High SD** (very loose) → `0.08` to `0.12`  

---

## 2. n_estimators (comes from cluster_size)

### Intuition:
“Bigger clusters have more structure → need more trees.”

### Suggested values:
- **cluster_size < 100** → `50–100` trees  
- **100 ≤ size ≤ 1000** → `150–200` trees  
- **size > 3000** → `250–350` trees  

---

## 3. max_samples (also from cluster_size)

### Intuition:
“Small cluster → smaller subsamples. Big cluster → bigger subsamples.”

### Suggested values:
- **cluster_size < 100** → `40–80%` of the cluster  
- **100 ≤ size ≤ 1000** → `60–90%` of the cluster  
- **size > 3000** → fixed cap like `512` or `1024` samples  

---

## Summary Mapping
- **cluster_sd → contamination**  
- **cluster_size → n_estimators**  
- **cluster_size → max_samples**  

These three settings adjust the Isolation Forest to each cluster’s personality.


----
### Final value tables
#### 1. Contamination rule: 



Contamination is chosen by first selecting a system-wide range:
- Tight system  →  0.005 to 0.03  
- Normal system →  0.01  to 0.08  
- Loose system  →  0.02  to 0.12  

The chosen range is stretched across k positions (k = number of clusters) to form a smooth scale of contamination values from minimum to maximum.  
Cluster SD ordering determines each cluster’s position on this scale: tighter SD maps to lower contamination, looser SD maps to higher contamination, and identical SD values map to the same position.


---

#### 2. n_estimators rule: 
| Cluster Size Range | Meaning      | Recommended n_estimators |
| ------------------ | ------------ | ------------------------ |
| **< 100**          | Very Small   | **50–80**                |
| **100–1000**       | Small–Medium | **150–200**              |
| **1000–3000**      | Medium–Large | **200–250**              |
| **> 3000**         | Very Large   | **250–350**              |


---

#### 3. max_samples rule:
| Cluster Size Range | Meaning      | Recommended max_samples                      |
| ------------------ | ------------ | -------------------------------------------- |
| **< 100**          | Very Small   | **40–80%** of cluster size                   |
| **100–1000**       | Small–Medium | **60–90%** of cluster size                   |
| **1000–3000**      | Medium–Large | **approx 90–95%** of cluster size            |
| **> 3000**         | Very Large   | **Fixed cap: 512–1024** (regardless of size) |



In [ ]:
def find_model_parameter(clust_sd, cluster_size):
    # df = scaled_df (just for this function)

    contamination_list, n_estimator_list, max_sample_list = [None]*len(clust_sd), [None]*len(clust_sd), [None]*len(clust_sd)

    # 1. Contamination (based on cluster SD)

    # Step 1: Compute SD spread
    spread = max(clust_sd) - min(clust_sd)
    
    # Step 2: Choose min/max contamination based on spread
    if spread <= 0.2:
        min_cont, max_cont = 0.005, 0.03
    elif spread <= 0.6:
        min_cont, max_cont = 0.01, 0.08
    else:
        min_cont, max_cont = 0.02, 0.12
    
    # Step 3: Generate contamination values (dynamic for any k)
    selected_sd_range = np.linspace(min_cont, max_cont, len(clust_sd))
    
    # Step 4: Group clusters by SD (equal SD → equal contamination)
    unique_sds = sorted(set(clust_sd))
    sd_to_rank = {sd: i for i, sd in enumerate(unique_sds)}
    
    # Step 5: Assign contamination using grouped ranks
    for i in range(len(clust_sd)):
        rank = sd_to_rank[clust_sd[i]] # which rank does the cluster with This SD has
        contamination_list[i] = selected_sd_range[rank]

    # ----------------------------------------------------------
    
    # 2. n_estimators &  max_samples (based on cluster size)

    for i in range(len(cluster_size)):
        if cluster_size[i] < 100: # Small sized cluster
            n_estimator_list[i] = 70
            max_sample_list[i] = int(cluster_size[i] * 0.5)
            
        elif cluster_size[i] <1000: # small to medium sized cluster
            n_estimator_list[i] = 150
            max_sample_list[i] = int(cluster_size[i] * 0.7)

        elif cluster_size[i] <3000: #  medium to large sized cluster
            n_estimator_list[i] = 200
            max_sample_list[i] = int(cluster_size[i] * 0.9)
            
        else:                     # Large sized cluster
            n_estimator_list[i] = 350
            max_sample_list[i] = 1000
     # ----------------------------------------------------------

    return contamination_list, n_estimator_list, max_sample_list
    
        

In [ ]:
contamination_list, n_estimator_list, max_sample_list = find_model_parameter(avg_sd, cluster_size)

In [ ]:
print(f"{'Cluster ID':<12} | {'Contamination':<15} | {'n_estimators':<12} | {'max_samples':<12}")
print("-" * 65)

for i in range(len(cluster_size)):
    print(f"{str(i):<12} | {contamination_list[i]:<15} | {n_estimator_list[i]:<12} | {max_sample_list[i]:<12}")


In [ ]:
cluster_size

In [ ]:
print(f"{'Cluster ID':<12} | {'Contamination':<15} | {'n_estimators':<12} | {'max_samples':<12}")
print("-" * 65)

for i in range(len(cluster_size)):
    print(f"{str(i):<12} | {contamination_list[i]:<15} | {n_estimator_list[i]:<12} | {max_sample_list[i]:<12}")